# Axis 2: Unified Three-System Probing

Probes verb-spatial binding in **three** generative systems on AGD20K (36 affordance categories):

1. **Flux** (text-to-image) — known-good baseline replicating Zhang et al. (H2a)
2. **Cosmos-Predict2-2B-Video2World** — base video diffusion (H2b)
3. **Cosmos-Policy-ALOHA-Predict2-2B** — manipulation-fine-tuned variant (H2c)

**Required:** Colab Pro with A100 (40GB) — both Flux and Cosmos require ≥24GB VRAM in BF16.

**Output:** results/tables/axis2_*_per_sample.csv and three_way_comparison artifacts. Commit these back to the repo on branch `nj-features` and the agent will pick them up.

**Workflow per system:** install → smoke-test → pilot run (30/category) → final run when satisfied. After all three systems are done, run the comparison cell.

## Cell 0 — Repo + dependencies

In [ ]:
!git clone https://github.com/aleksantari/VLA-affordance.git
%cd VLA-affordance
!git checkout nj-features
!git pull origin nj-features
!pip install -e . -q
!pip install -r requirements_axis2.txt -q
print('\n✓ Setup complete')

## Cell 1 — GPU + dataset

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Runtime > Change runtime type > A100')
name = torch.cuda.get_device_name()
vram = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f'GPU: {name}  VRAM: {vram:.1f} GB')
if vram < 35:
    print('⚠ Cosmos needs ≥32 GB; Flux needs ≥24 GB. Use --cpu_offload if low VRAM.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_AGD20K_PATH = '/content/drive/MyDrive/datasets/AGD20K'
import os
if os.path.exists(DRIVE_AGD20K_PATH):
    !python data/download_agd20k.py --from_drive {DRIVE_AGD20K_PATH} --output_dir ./data/agd20k
else:
    print(f'Drive path missing: {DRIVE_AGD20K_PATH} — falling back to direct download.')
    !python data/download_agd20k.py --output_dir ./data/agd20k
!python data/download_agd20k.py --validate_only --output_dir ./data/agd20k

## Cell 2 — System A: Flux (H2a)

Smoke test, then pilot, then final. Pilot: 30 samples/category × 36 categories ≈ 1080 samples → ~30-45 min on A100 with schnell.

In [ ]:
!python scripts/09_setup_flux.py --model schnell

In [ ]:
# Pilot — 30 samples per category, schnell (fast)
!python scripts/10_run_interaction_probing.py \
    --model schnell \
    --max_per_category 30 \
    --save_attention_maps

In [ ]:
# Final — full dataset, dev model (publication quality). UNCOMMENT WHEN READY.
# !python scripts/10_run_interaction_probing.py --model dev --save_attention_maps

## Cell 3 — System B: Cosmos-Predict2 V2W (H2b)

Base video diffusion model. Conditioned on first-frame image (the AGD20K image itself) + text prompt. Cross-attention extracted via `interaction.cosmos_attention.CosmosVerbAttentionExtractor`.

In [ ]:
# Quick smoke test on a small subset
!python scripts/10b_run_cosmos_probing.py \
    --system cosmos_predict2_v2w \
    --categories cut hold pour \
    --max_per_category 3 \
    --num_inference_steps 8

In [ ]:
# Pilot — 30/category. ~1.5-2.5 hours on A100.
!python scripts/10b_run_cosmos_probing.py \
    --system cosmos_predict2_v2w \
    --max_per_category 30 \
    --num_inference_steps 12

## Cell 4 — System C: Cosmos Policy (H2c)

Manipulation-fine-tuned variant of Cosmos-Predict2-2B-Video2World. Same architecture, different weights — a clean H2b/H2c paired comparison.

**Note:** Cosmos Policy was fine-tuned with proprio + multi-view inputs. We pass the AGD20K image as the conditioning frame and rely on cross-attention only seeing text↔latent (proprio is a separate stream). This is methodologically sound for verb-spatial binding probing because the cross-attention does not change shape regardless of proprio.

In [ ]:
!python scripts/10b_run_cosmos_probing.py \
    --system cosmos_policy \
    --categories cut hold pour \
    --max_per_category 3 \
    --num_inference_steps 8

In [ ]:
# Pilot — 30/category, paired with Cosmos Predict2 V2W (same sample IDs)
!python scripts/10b_run_cosmos_probing.py \
    --system cosmos_policy \
    --max_per_category 30 \
    --num_inference_steps 12

## Cell 5 — Three-way comparison + hypothesis tests

In [ ]:
!python scripts/12_three_way_comparison.py

In [ ]:
# Display the headline figures
from IPython.display import Image, display
from pathlib import Path
for f in sorted(Path('./results/figures/axis2').glob('three_way_*.png')):
    print(f.name); display(Image(str(f), width=900))

# Print hypothesis test results
import json, pprint
with open('./results/tables/axis2_hypothesis_tests.json') as fp:
    pprint.pp(json.load(fp))

## Cell 6 — Commit results back to the repo

After running all three systems, commit the per-sample CSVs + summary artifacts back to `nj-features` so the agent can pick them up locally and write the paper.

In [ ]:
# Configure git in Colab — replace with your name/email
!git config user.email 'jajda.n@northeastern.edu'
!git config user.name 'nitik1998 (Colab)'
!git add results/tables results/figures/axis2
!git status
!git commit -m 'research(results): Axis 2 pilot — Flux + Cosmos Predict2 V2W + Cosmos Policy on AGD20K' -m 'Per-sample CSVs and three-way comparison artifacts. 30/category pilot.'
!git push origin nj-features

## Cell 7 — Backup to Drive (optional)

Useful as a redundant backup in case the git push fails or you need the cached attention maps later (they can be large).

In [ ]:
import shutil
from pathlib import Path
drive_results = Path('/content/drive/MyDrive/VLA-affordance-results/axis2')
drive_results.mkdir(parents=True, exist_ok=True)
shutil.copytree('./results/tables', str(drive_results / 'tables'), dirs_exist_ok=True)
shutil.copytree('./results/figures/axis2', str(drive_results / 'figures'), dirs_exist_ok=True)
for s in ('flux_attention', 'cosmos_predict2_v2w_attention', 'cosmos_policy_attention'):
    src = Path('./results/cached_features') / s
    if src.exists() and any(src.iterdir()):
        shutil.copytree(str(src), str(drive_results / s), dirs_exist_ok=True)
print('✓ Backup complete:', drive_results)